In [ ]:
# ========================================================
# CELL 1 & 2: RECONFIGURED DEPENDENCY INSTALLATION & SEEDS
# ========================================================
!pip install -q transformers accelerate pandas numpy torch nltk bert-score

import os
import time
import torch
import pandas as pd
import numpy as np
import random
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Deterministic setups
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Compute Target Engine: {device}")

In [ ]:
import os
import pandas as pd
from google.colab import drive

# 1. Ensure Google Drive is securely connected
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# 2. Set the exact folder path found in your My Drive
DATA_DIR = "/content/drive/MyDrive/NLU_Sa_En_Datasets"
print(f"🎯 Data engine directory target successfully set to: '{DATA_DIR}'")

# 3. Dynamic split loader function
def load_and_align_split(split):
    if split == "train":
        suffix = "_10000.csv"
    elif split in ["dev", "test"]:
        suffix = "_1000.csv"
    else:
        suffix = ".csv"

    sa_path = os.path.join(DATA_DIR, f"{split}_sa{suffix}")
    en_path = os.path.join(DATA_DIR, f"{split}_en{suffix}")

    # Fallback to check standard names without numbers
    if not os.path.exists(sa_path):
        sa_path = os.path.join(DATA_DIR, f"{split}_sa.csv")
        en_path = os.path.join(DATA_DIR, f"{split}_en.csv")

    if not os.path.exists(sa_path):
        raise FileNotFoundError(f"Missing file path: {sa_path}")

    print(f"Loading {split} from: {sa_path}")
    df_sa = pd.read_csv(sa_path)

    if os.path.exists(en_path):
        df_en = pd.read_csv(en_path)
        merged = pd.merge(df_sa, df_en, on="Source_id", how="inner")
        return merged.dropna(subset=["Sentence_sa", "Sentence_en"])
    else:
        print(f"ℹ️ Target alignment hidden for [{split}]. Building blind inference structure.")
        df_sa = df_sa.dropna(subset=["Sentence_sa"])
        df_sa["Sentence_en"] = ""
        return df_sa
# 4. Load dataset splits smoothly
try:
    train_df = load_and_align_split("train")
    dev_df = load_and_align_split("dev")
    test_df = load_and_align_split("test")
    print(f"\n🎉 Success! Loaded train ({len(train_df)}), dev ({len(dev_df)}), and test ({len(test_df)}) data rows!")
except Exception as e:
    print(f"\n❌ Setup Error: {e}")


In [ ]:
# ========================================================
# LOADING PRETRAINED MODEL
# ========================================================
MODEL_NAME = "facebook/nllb-200-distilled-600M"

print(f"Loading pretrained tokenizer and model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# Define language codes for Sanskrit (Devanagari) and English
SRC_LANG = "san_Deva"
TGT_LANG = "eng_Latn"

print("✅ Pretrained NLLB-200 model loaded successfully!")
total_params = sum(p.numel() for p in model.parameters())
print(f"Model size checkpoint: {total_params:,} parameters.")

In [ ]:
def execute_pretrained_translation(model, tokenizer, raw_sanskrit_string, max_length=128):
    model.eval()
    tokenizer.src_lang = SRC_LANG

    inputs = tokenizer(raw_sanskrit_string, return_tensors="pt", padding=True, truncation=True).to(device)
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_length=max_length,
            num_beams=5,               # Increased from 4 to 5 for deeper search paths
            length_penalty=1.0,        # Encourages matching optimal sentence lengths
            no_repeat_ngram_size=3,    # Eliminates repetitive phrases/loops
            early_stopping=True
        )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

In [ ]:
# ========================================================
# SCORING CELL
# ========================================================
from nltk.translate.bleu_score import corpus_bleu
from bert_score import score as bert_score_fn

print("--- RUNNING OPTIMIZED PRETRAINED MODEL INFERENCE ---")
inference_start_timestamp = time.time()
test_predictions_pool = []

print(f"Translating {len(test_df)} sentences from the test split via NLLB-200...")
for index, row in test_df.iterrows():
    # CHANGE 1: Clean up common formatting anomalies before feeding to the tokenizer
    sanskrit_sentence = str(row['Sentence_sa']).strip().replace("।", "").replace("॥", "")

    output_phrase = execute_pretrained_translation(model, tokenizer, sanskrit_sentence)
    test_predictions_pool.append(output_phrase)

elapsed_generation_seconds = time.time() - inference_start_timestamp

print("\n========================================================")
print("                    EFFICIENCY PROFILE                  ")
print("========================================================")
print(f"Total Pretrained Model Parameters : {total_params:,}")
print(f"Total Test Set Inference Speed    : {elapsed_generation_seconds:.2f} seconds")
print("========================================================\n")

gold_references = test_df["Sentence_en"].astype(str).tolist()

if gold_references and gold_references[0] != "":
    # CHANGE 2: Standardize and lowercase strings right before evaluation to eliminate strict-case penalties
    test_predictions_cleaned = [pred.strip().lower() for pred in test_predictions_pool]
    gold_references_cleaned = [ref.strip().lower() for ref in gold_references]

    ref_bleu = [[ref.split()] for ref in gold_references_cleaned]
    hyp_bleu = [pred.split() for pred in test_predictions_cleaned]

    internal_bleu = corpus_bleu(ref_bleu, hyp_bleu)
    print(f"Calculated Optimized Pretrained BLEU Score: {internal_bleu:.4f}")

    P, R, F1 = bert_score_fn(test_predictions_cleaned, gold_references_cleaned, lang="en", rescale_with_baseline=True)
    print(f"Calculated Optimized Pretrained BERTScore F1: {F1.mean().item():.4f}")
else:
    print("ℹ️ Private hidden evaluation run: skipping accuracy metrics.")

# Compile Final Submission File
submission_export_dataframe = pd.DataFrame({
    'Source_id': test_df['Source_id'].tolist(),
    'Sentence_en': test_predictions_pool  # Saves the original case translations to file
})

submission_export_dataframe.to_csv("submission.csv", index=False, encoding="utf-8")
print("\n🎉 Success! Optimized pretrained submission output file saved as 'submission.csv'.")